# Demo 04 : A workflow or an agent

Run the cells in order and watch. There is nothing to write.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph-Student/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


In [2]:
import json
import re
import sqlite3
from pathlib import Path

from langchain.tools import tool

DB = data("game-studio/studio.db")
SECTIONS = []
for path in sorted(Path(data_folder("game-studio/docs")).glob("*.md")):
    title, *parts = path.read_text(encoding="utf-8").split("\n## ")
    SECTIONS += [f"{title.splitlines()[0]}\n## {part}" for part in parts]


def rows(sql, *args):
    with sqlite3.connect(DB) as conn:
        conn.row_factory = sqlite3.Row
        return [dict(row) for row in conn.execute(sql, args)]


def words(text):
    return set(re.findall(r"[a-z0-9-]{3,}", text.lower()))


def best(texts, query, k):
    q = words(query)
    return sorted(texts, key=lambda t: -len(q & words(t)))[:k]

We define six tools over the data, a meter for calls and tokens, `look_up_bug`, which reads the bug and the latest
build in plain Python, and four playtester reports.

In [3]:
@tool
def game_info(title: str) -> str:
    """Look up a Comet Arcade game by its title or id. Returns id, title, genre, status and pitch."""
    t = title.strip()
    found = rows("SELECT game_id, title, genre, status, pitch FROM games WHERE game_id = ? OR title LIKE ?", t.upper(), f"%{t}%")
    return json.dumps(found[0]) if found else "Unknown game"

@tool
def bugs_for_game(game_id: str, status: str = "open") -> str:
    """List bugs for a game id such as CS-01. Status is open, fixed or all."""
    status = status.strip().lower()
    return json.dumps(rows("SELECT bug_id, title, severity, status, found_by FROM bugs WHERE game_id = ? AND (? = 'all' OR status = ?)",
                           game_id.strip().upper(), status, status))

@tool
def bug_details(bug_id: str) -> str:
    """Read one bug by id such as BUG-101, with severity, status and notes."""
    found = rows("SELECT * FROM bugs WHERE bug_id = ?", bug_id.strip().upper())
    return json.dumps(found[0]) if found else "Unknown bug"

@tool
def builds_for_game(game_id: str) -> str:
    """List the builds of a game id such as CS-01, with tests passed and status."""
    return json.dumps(rows("SELECT build, twist, tests_passed, tests_total, status, notes FROM builds WHERE game_id = ? ORDER BY build",
                           game_id.strip().upper()))

@tool
def assets_for_game(game_id: str) -> str:
    """List sprites, palettes and sounds for a game id such as CS-01, with colour and status."""
    return json.dumps(rows("SELECT asset_id, kind, name, colour, status FROM assets WHERE game_id = ?", game_id.strip().upper()))

@tool
def search_design_docs(query: str) -> str:
    """Search the design docs and the studio style guide. Returns the three best matching sections."""
    return "\n\n---\n\n".join(best(SECTIONS, query, 3))

from langchain_core.callbacks import BaseCallbackHandler


class Meter(BaseCallbackHandler):
    def __init__(self):
        self.calls = self.input = self.output = 0

    def on_llm_end(self, response, **kwargs):
        usage = response.generations[0][0].message.usage_metadata or {}
        self.calls += 1
        self.input += usage.get("input_tokens", 0)
        self.output += usage.get("output_tokens", 0)

    def __str__(self):
        return f"{self.calls} calls, {self.input:,} input tokens, {self.output:,} output tokens"


def correct(answer, expected):
    text = answer.lower()
    return all(any(alt in text for alt in item.split("|")) for item in expected)

def look_up_bug(bug_id: str) -> dict:
    bug = rows("SELECT * FROM bugs WHERE bug_id = ?", bug_id)[0]
    game = rows("SELECT title, status FROM games WHERE game_id = ?", bug["game_id"])[0]
    builds = rows("SELECT build, tests_passed, tests_total, status, notes FROM builds WHERE game_id = ? ORDER BY build",
                  bug["game_id"])
    return {"bug": bug, "game": game, "latest_build": builds[-1]}

REPORTS = [
    "BUG-104 on Comet Snake again : I picked up the shield and nothing happened.",
    "Comet Snake, BUG-105 : two comets, and the hit sound played twice.",
    "Star Slide BUG-202. A slide into the rock next to my ship still counted.",
    "I keep losing Comet Snake to comets I cannot see. I have no bug number.",
]
NOTE = ("Write a three-line note for the Coder or the Artist : what happens, the likely cause, "
        "and the first thing to change.")

## a · workflow-or-agent

We fix the workflow's path in code. A regular expression finds the bug id, `look_up_bug` reads the database, and the
model writes the note in one model call per report. A report without a bug id goes to the Producer.

In [4]:
from typing import Literal, TypedDict

from langgraph.graph import END, START, StateGraph


class Ticket(TypedDict, total=False):
    report: str
    bug_id: str | None
    facts: dict
    note: str


def parse(state: Ticket) -> dict:
    found = re.search(r"BUG-\d{3}", state["report"])
    return {"bug_id": found and found.group()}


def route(state: Ticket) -> Literal["look_up", "escalate"]:
    return "look_up" if state["bug_id"] else "escalate"


def look_up(state: Ticket) -> dict:
    return {"facts": look_up_bug(state["bug_id"])}


def write_note(state: Ticket) -> dict:
    return {"note": llm.invoke(f"{NOTE}\n\nReport : {state['report']}\nFacts : {state['facts']}").content}


builder = StateGraph(Ticket)
builder.add_node("parse", parse)
builder.add_node("look_up", look_up)
builder.add_node("write_note", write_note)
builder.add_node("escalate", lambda state: {"note": "No bug id : sent to the Producer."})
builder.add_edge(START, "parse")
builder.add_conditional_edges("parse", route)
builder.add_edge("look_up", "write_note")
builder.add_edge("write_note", END)
builder.add_edge("escalate", END)
workflow = builder.compile()
print(workflow.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	parse(parse)
	look_up(look_up)
	write_note(write_note)
	escalate(escalate)
	__end__([<p>__end__</p>]):::last
	__start__ --> parse;
	look_up --> write_note;
	parse -.-> escalate;
	parse -.-> look_up;
	escalate --> __end__;
	write_note --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [5]:
runs = {}
for report in REPORTS:
    meter = Meter()
    note = workflow.invoke({"report": report}, {"callbacks": [meter]})["note"]
    runs[report] = {"workflow": meter}
    print(f"> {report}\n{note}\n[{meter}]\n")

> BUG-104 on Comet Snake again : I picked up the shield and nothing happened.
What happens: The shield pickup stays on the board and grants no shield.
Likely cause: The pickup collision isn’t being handled, so the shield flag never changes.
First thing to change: Check the pickup collision logic and set the shield flag when collected.
[1 calls, 184 input tokens, 56 output tokens]



> Comet Snake, BUG-105 : two comets, and the hit sound played twice.
What happens: Two comets land on the same tick, and the hit sound plays twice.
Likely cause: The sound is triggered once per comet instead of once per tick.
First thing to change: Play the hit sound only once when handling a tick with one or more comet hits.
[1 calls, 186 input tokens, 62 output tokens]



> Star Slide BUG-202. A slide into the rock next to my ship still counted.
Coder: A slide into the rock next to the ship still increments the slide counter.
Likely cause: The counter updates when a slide is attempted, before the blocked move is checked.
First change: Increment the counter only after the ship successfully moves.
[1 calls, 187 input tokens, 54 output tokens]

> I keep losing Comet Snake to comets I cannot see. I have no bug number.
No bug id : sent to the Producer.
[0 calls, 0 input tokens, 0 output tokens]



The agent gets the six tools and the same instruction. It decides what to look up and how often.

In [6]:
from langchain.agents import create_agent

agent = create_agent(llm, tools=[game_info, bugs_for_game, bug_details, builds_for_game, assets_for_game, search_design_docs],
                     system_prompt=f"You are the bug desk of Comet Arcade. {NOTE} Use the tools.")

for report in REPORTS:
    meter = Meter()
    messages = agent.invoke({"messages": [{"role": "user", "content": report}]}, {"callbacks": [meter]})["messages"]
    runs[report]["agent"] = meter
    print(f"> {report}\ntools: {[m.name for m in messages if m.type == 'tool']}")
    print(f"{messages[-1].content}\n[{meter}]\n")

> BUG-104 on Comet Snake again : I picked up the shield and nothing happened.
tools: ['bug_details', 'game_info', 'search_design_docs']
Coder — Picking up the shield leaves it on the board and the shield flag stays false.  
Likely cause — Shield extras aren’t being handled when the snake head lands on them.  
First change — In the head/extra collision logic, set `shield` to true and remove the collected shield extra.
[4 calls, 1,942 input tokens, 139 output tokens]



> Comet Snake, BUG-105 : two comets, and the hit sound played twice.
tools: ['game_info', 'bug_details', 'search_design_docs', 'builds_for_game']
**What happens:** When both comets land on the snake in the same tick, the hit sound plays twice.  
**Likely cause:** The collision handler plays the sound once for each comet hit.  
**First thing to change:** Guard the hit sound so it plays only once per tick.
[3 calls, 1,634 input tokens, 171 output tokens]



> Star Slide BUG-202. A slide into the rock next to my ship still counted.
tools: ['bug_details', 'game_info', 'builds_for_game', 'search_design_docs']
**Coder — What:** Sliding into a rock without moving still increments the slide count.  
**Likely cause:** The counter increments when a slide is attempted, before checking whether the ship moved.  
**First change:** Increment the count only after confirming the ship’s position changed.
[3 calls, 1,546 input tokens, 163 output tokens]



> I keep losing Comet Snake to comets I cannot see. I have no bug number.
tools: ['game_info', 'bugs_for_game', 'bug_details']
What happens: Comets are hard to see against the dark background, making them difficult to dodge.  
Likely cause: The comet sprite (CS-SPR-03) has too little contrast with the background.  
First thing to change: Artist—adjust the comet sprite’s colors to make it stand out.
[4 calls, 1,919 input tokens, 136 output tokens]



On the three reports with a bug id, both usually reach the same bug, and the workflow does it with one model call. On
the last report the workflow escalates because its path has no step for a report without an id. The agent can search
the bugs, the assets and the style guide instead. Whether it finds the sprite varies between runs, and the cost is
several calls.

In [7]:
print(f"{'report':40} {'workflow':>16} {'agent':>16}")
for report, r in runs.items():
    w, a = r["workflow"], r["agent"]
    print(f"{report[:40]:40} {w.calls:>3} calls {w.input:>6,} {a.calls:>3} calls {a.input:>6,}")

report                                           workflow            agent
BUG-104 on Comet Snake again : I picked    1 calls    184   4 calls  1,942
Comet Snake, BUG-105 : two comets, and t   1 calls    186   3 calls  1,634
Star Slide BUG-202. A slide into the roc   1 calls    187   3 calls  1,546
I keep losing Comet Snake to comets I ca   0 calls      0   4 calls  1,919


## Things to try

Optional.

- Give the workflow one agentic step, a node that calls the agent for reports without a bug id
- Write a report with a typo in the bug id and run both